# Data Loading & Cleaning

This notebook is the **one-time bootstrap** for the dataset.

- **First run:** loads from HuggingFace, cleans, saves `noise_cleaned.parquet`
- **Subsequent runs:** skips loading entirely — `noise_cleaned.parquet` is maintained by `00_ingest.ipynb` going forward

The neighborhood coordinates CSV is always regenerated from whatever is in the parquet,
so it stays up to date as new data is ingested.

## Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEANED_PARQUET   = "../data/processed/noise_cleaned.parquet"
NEIGHBORHOODS_CSV = "../data/raw/neighborhoods.csv"
LAST_INGESTED     = "../data/processed/last_ingested.txt"

## Step 1: Check Whether Bootstrap Is Needed

In [2]:
parquet_exists = Path(CLEANED_PARQUET).exists()

if parquet_exists:
    df = pd.read_parquet(CLEANED_PARQUET)
    print(f"noise_cleaned.parquet already exists — skipping HuggingFace load.")
    print(f"  Rows: {len(df):,}")
    print(f"  Date range: {pd.to_datetime(df['Created Date']).min().date()} "
          f"→ {pd.to_datetime(df['Created Date']).max().date()}")
    print()
    print("To add new data, run 00_ingest.ipynb instead.")
else:
    print("No parquet found — running first-time bootstrap from HuggingFace.")

noise_cleaned.parquet already exists — skipping HuggingFace load.
  Rows: 4,842,324
  Date range: 2020-01-01 → 2026-05-22

To add new data, run 00_ingest.ipynb instead.


## Step 2: First-Time Load from HuggingFace

Only runs if `noise_cleaned.parquet` does not exist yet.
The HuggingFace dataset covers 2020 onward. After this runs once,
use `00_ingest.ipynb` to pull anything newer.

In [3]:
if not parquet_exists:
    from datasets import load_dataset

    print("Loading 311 Noise Complaints from HuggingFace...")
    print("(Large file — may take ~2 minutes)")

    ds = load_dataset("idakam/311-nyc-noise-complaints", split="train")
    df = ds.to_pandas()
    print(f"Loaded {len(df):,} rows")

    neighborhoods = pd.read_csv(NEIGHBORHOODS_CSV)

    # ── Datetime & temporal features ──────────────────────────────────────
    df['Created Date'] = pd.to_datetime(df['Created Date'])
    df['Year']         = df['Created Date'].dt.year
    df['Month']        = df['Created Date'].dt.month
    df['Week']         = df['Created Date'].dt.isocalendar().week
    df['Day_of_Week']  = df['Created Date'].dt.dayofweek
    df['Day_Name']     = df['Created Date'].dt.day_name()
    df['Hour']         = df['Created Date'].dt.hour
    df['Date']         = df['Created Date'].dt.date

    # ── Time buckets ──────────────────────────────────────────────────────
    def get_time_bucket(hour):
        if 6 <= hour < 12:        return 'morning'
        elif 12 <= hour < 18:     return 'afternoon'
        elif 18 <= hour < 22:     return 'evening'
        elif hour >= 22 or hour < 2: return 'night'
        else:                     return 'overnight'

    df['Time_Bucket'] = df['Hour'].apply(get_time_bucket)

    # ── Season ────────────────────────────────────────────────────────────
    df['Season'] = df['Month'].map({
        12: 'Winter', 1: 'Winter', 2: 'Winter',
         3: 'Spring', 4: 'Spring', 5: 'Spring',
         6: 'Summer', 7: 'Summer', 8: 'Summer',
         9: 'Fall',  10: 'Fall',  11: 'Fall'
    })

    # ── Clean ZIPs & boroughs ─────────────────────────────────────────────
    df['Incident Zip']       = df['Incident Zip'].astype(str).str.strip().str[:5]
    neighborhoods['ZipCode'] = neighborhoods['ZipCode'].astype(str).str.strip().str[:5]
    df['Borough']            = df['Borough'].str.upper()
    neighborhoods['Borough'] = neighborhoods['Borough'].str.upper()
    df = df[df['Borough'] != 'UNSPECIFIED'].copy()

    # ── Merge neighborhood lookup ─────────────────────────────────────────
    df['Borough_311'] = df['Borough']
    neighborhoods_renamed = neighborhoods.rename(columns={'Borough': 'Borough_Zip'})

    df = df.merge(
        neighborhoods_renamed[['ZipCode', 'Neighborhood', 'Borough_Zip']],
        left_on='Incident Zip',
        right_on='ZipCode',
        how='left'
    )
    df['Borough'] = df['Borough_Zip'].fillna(df['Borough_311'])
    df.drop(columns=['ZipCode', 'Borough_311', 'Borough_Zip'], inplace=True, errors='ignore')

    # ── Save ──────────────────────────────────────────────────────────────
    df.to_parquet(CLEANED_PARQUET, index=False)
    print(f"\n✓ Saved noise_cleaned.parquet — {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Date range: {df['Created Date'].min().date()} → {df['Created Date'].max().date()}")

    # Write last_ingested so 00_ingest knows where to pick up from
    latest = df['Created Date'].max().strftime("%Y-%m-%dT%H:%M:%S")
    with open(LAST_INGESTED, 'w') as f:
        f.write(latest)
    print(f"  last_ingested.txt set to: {latest}")
    print()
    print("Bootstrap complete. Run 00_ingest.ipynb to pull records newer than this.")

## Step 3: Data Quality Checks

In [4]:
mapped_pct = df['Neighborhood'].notna().mean()
print(f"Neighborhood mapped: {mapped_pct:.1%}")

multi_borough = (
    df.groupby('Neighborhood')['Borough']
    .nunique()
    .sort_values(ascending=False)
)
print("\nNeighborhoods spanning >1 borough (should be 0 after ZIP correction):")
print(multi_borough[multi_borough > 1].head())

print(f"\nTime bucket distribution:")
print(df['Time_Bucket'].value_counts())

Neighborhood mapped: 99.2%

Neighborhoods spanning >1 borough (should be 0 after ZIP correction):
Series([], Name: Borough, dtype: int64)

Time bucket distribution:
Time_Bucket
night        1773118
evening      1174888
afternoon     858436
morning       543304
overnight     492578
Name: count, dtype: int64


## Step 4: Regenerate Neighborhood Coordinates

Always regenerated from the current parquet so it stays up to date
as new ingested data adds coverage.

In [5]:
neighborhood_coords = df.groupby(['Borough', 'Neighborhood']).agg(
    Latitude  = ('Latitude',  'mean'),
    Longitude = ('Longitude', 'mean')
).reset_index()

neighborhood_coords = neighborhood_coords.dropna(subset=['Neighborhood', 'Latitude', 'Longitude'])
neighborhood_coords = neighborhood_coords[neighborhood_coords['Borough'] != 'UNKNOWN']

neighborhood_coords.to_csv("../data/processed/neighborhood_coordinates.csv", index=False)
print(f"✓ Saved neighborhood_coordinates.csv — {len(neighborhood_coords):,} neighborhoods")

✓ Saved neighborhood_coordinates.csv — 42 neighborhoods
